# Qwirkle AlphaZero Training on Kaggle

Trains a Graph Transformer + MCTS bot for Qwirkle on free Kaggle GPU.

**Pipeline**:
1. Setup Rust + use system PyTorch as libtorch backend
2. Clone repo & build binaries
3. Generate self-play data (Greedy vs Greedy bootstrap)
4. Train large model (NetConfig::LARGE — 6 layers, d_model=128)
5. AlphaZero loop with ISMCTS (5 samples × 200 MCTS sims)
6. Evaluate vs Greedy baseline

**Settings**: GPU T4 x2 (or P100), ~6-10h total runtime.

## 1. Environment check + paths setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

In [ ]:
import os, torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

# Set up paths used by all subsequent cells
TORCH_DIR = os.path.dirname(torch.__file__)
TORCH_LIB = os.path.join(TORCH_DIR, 'lib')
os.environ['LIBTORCH'] = TORCH_DIR
os.environ['LIBTORCH_USE_PYTORCH'] = '1'
os.environ['LIBTORCH_BYPASS_VERSION_CHECK'] = '1'
os.environ['LD_LIBRARY_PATH'] = f"{TORCH_LIB}:" + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['PATH'] = f"{os.environ['HOME']}/.cargo/bin:" + os.environ['PATH']

# Persist for %%bash cells
%env TORCH_LIB={TORCH_LIB}
%env LD_LIBRARY_PATH={TORCH_LIB}
%env LIBTORCH_USE_PYTORCH=1
%env LIBTORCH_BYPASS_VERSION_CHECK=1

print('TORCH_LIB:', TORCH_LIB)
print('LD_LIBRARY_PATH:', os.environ['LD_LIBRARY_PATH'])
print()
print('Files in TORCH_LIB:')
!ls $TORCH_LIB | head -10

## 2. Install Rust

In [ ]:
%%bash
if ! command -v rustc &> /dev/null; then
    curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
fi
source $HOME/.cargo/env
rustc --version && cargo --version

## 3. Clone repo & build

In [ ]:
%%bash
cd /kaggle/working
if [ ! -d qwirkle ]; then
    git clone https://github.com/specialjcg/qwirkle.git
fi
cd qwirkle && git checkout dev && git pull

In [ ]:
%%bash
source $HOME/.cargo/env
export LIBTORCH_USE_PYTORCH=1
export LIBTORCH_BYPASS_VERSION_CHECK=1
cd /kaggle/working/qwirkle/backend
# Show last 30 lines so any error is visible
cargo build --features neural --release 2>&1 | tail -30

In [ ]:
%%bash
ls -lh /kaggle/working/qwirkle/backend/target/release/{selfplay,train_bot,alphazero,evaluate} 2>&1

## 4. Generate self-play data

3000 games of greedy-vs-greedy with bag-aware features and long-term reward shaping.
Output ~2 GB. Takes ~10-15 min.

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
mkdir -p /kaggle/working/data /kaggle/working/models
./target/release/selfplay \
    --games 3000 \
    --out /kaggle/working/data/v6_bootstrap.bin \
    --epsilon 0.1 \
    2>&1 | tail -20

In [ ]:
!ls -lh /kaggle/working/data/

## 5. Train large model

NetConfig::LARGE: 6 layers, d_model=128, 8 heads (~5.6M params).
100 epochs with cosine LR schedule + reward shaping. ~30-60 min on T4.

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
./target/release/train_bot \
    --data /kaggle/working/data/v6_bootstrap.bin \
    --epochs 100 \
    --batch 256 \
    --lr 0.001 \
    --out /kaggle/working/models/v6_large.pt \
    --patience 20 \
    --max-samples 200000 \
    --policy-weight 0.5 \
    --large \
    2>&1 | tail -30

In [ ]:
!ls -lh /kaggle/working/models/

## 6. Quick eval of base model vs Greedy

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
./target/release/evaluate \
    --model /kaggle/working/models/v6_large.pt \
    --games 50 \
    --large \
    2>&1 | tail -10

## 7. AlphaZero loop with ISMCTS

Iterates: self-play with MCTS → train candidate → arena vs Greedy → keep if better.

**Settings**:
- 30 iterations max
- 60 self-play games per iter (mixed: 50% neural-vs-neural + 50% neural-vs-greedy)
- 200 MCTS sims per move
- 5 ISMCTS determinizations
- 40 arena games vs Greedy as acceptance test
- Stop when win rate vs Greedy ≥ 75%

Estimated runtime: 4-8h depending on GPU.

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
./target/release/alphazero \
    --iterations 30 \
    --selfplay-games 60 \
    --mcts-sims 200 \
    --ismcts 5 \
    --arena-games 40 \
    --eval-every 2 \
    --eval-games 40 \
    --target-greedy-winrate 0.75 \
    --win-threshold 0.55 \
    --out-dir /kaggle/working/models/az_v7 \
    --init-model /kaggle/working/models/v6_large.pt \
    --large \
    2>&1 | tail -50

## 8. Final evaluation

In [ ]:
%%bash
export LD_LIBRARY_PATH=$TORCH_LIB:$LD_LIBRARY_PATH
cd /kaggle/working/qwirkle/backend
echo '=== Best AlphaZero model vs Greedy (value-only) ==='
./target/release/evaluate \
    --model /kaggle/working/models/az_v7/best.pt \
    --games 100 \
    --large \
    2>&1 | tail -10

echo ''
echo '=== Best AlphaZero model vs Greedy (with MCTS 200) ==='
./target/release/evaluate \
    --model /kaggle/working/models/az_v7/best.pt \
    --games 50 \
    --mcts 200 \
    --large \
    2>&1 | tail -10

## 9. Save model as Kaggle output (persistent)

Files in `/kaggle/working/` are kept after the session ends and downloadable from the notebook output.

In [ ]:
%%bash
echo '=== Final models ==='
ls -lh /kaggle/working/models/v6_large.pt 2>/dev/null
ls -lh /kaggle/working/models/az_v7/best.pt 2>/dev/null

find /kaggle/working/models/az_v7 -name 'iter*_samples.bin' -delete 2>/dev/null
find /kaggle/working/models/az_v7 -name 'iter*_candidate.pt' -delete 2>/dev/null

du -sh /kaggle/working/models /kaggle/working/data 2>/dev/null

In [ ]:
import os
if os.path.exists('/kaggle/working/data/v6_bootstrap.bin'):
    size_gb = os.path.getsize('/kaggle/working/data/v6_bootstrap.bin') / 1e9
    print(f"Removing data file ({size_gb:.1f} GB)")
    os.remove('/kaggle/working/data/v6_bootstrap.bin')
    print('Done.')